In [233]:
import sys

print(sys.executable)

C:\Users\jugal\langchain-env\Scripts\python.exe


# Note:
1. First we will create a model which can search on Internet
2. Then we will add some more tools also

In [234]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests

from dotenv import load_dotenv

In [235]:
load_dotenv()

True

In [123]:
from langchain_tavily import TavilySearch
from langchain_community.tools import DuckDuckGoSearchRun

In [124]:
search = DuckDuckGoSearchRun()

res = search.invoke("Who is MS Dhoni?")

C:\Users\jugal\langchain-env\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


In [71]:
print(res)

Dhoni was born on 7 July 1981 in Ranchi, Bihar (now in Jharkhand) in a Hindu Rajput family to Pan Singh and Devaki Devi. [2][3] His parents … MS Dhoni is a legendary figure in the world of cricket. Known for his outstanding leadership, calm demeanor under pressure, and … Feb 23, 2025 · Mahendra Singh Dhoni, fondly known as MS Dhoni, is one of the most successful and beloved cricketers in the … Mar 24, 2025 · Mahendra Singh Dhoni, popularly known as MS Dhoni, is one of the greatest cricket captains and wicketkeeper-batsmen in … Aug 19, 2025 · Discover MS Dhoni’s untold journey before cricket fame—his childhood, railway job, early struggles, and the moments …


In [72]:
import os

os.environ['SERPER_API_KEY'] = "0a4bee2f4a7701bc4dc73d4f0071260490e16094"

In [136]:
from langchain_community.utilities import GoogleSerperAPIWrapper

serper_search = GoogleSerperAPIWrapper(type = 'news')

results = serper_search.results("What is the latest news on rain in Gujarat, India?")


In [100]:
for result in results['news']:

    print(f"---News:{result['position']}---")
    print()

    print(result['title'])
    print()

    print(result['date'])
    print()

    print(result['snippet'])
    print()

    print(result['link'])
    print()

    print('----------------------------------------------------------------------------------------------------------------------------------')


---News:1---

Aug brings 21% seasonal rain to Gujarat; 15% in last week alone

11 hours ago

Ahmedabad: In the first 24 days of Aug this year, the state received 21% of average seasonal rainfall. Out of the total, 15% was recorded in...

https://timesofindia.indiatimes.com/city/ahmedabad/aug-brings-21-seasonal-rain-to-gujarat-15-in-last-week-alone/articleshow/123511020.cms

----------------------------------------------------------------------------------------------------------------------------------
---News:2---

Heavy rainfall to continue over northwest India; Gujarat and Rajasthan to see extended showers: IMD

1 hour ago

Northwest India is likely to experience heavy to very heavy rainfall until August 26, with conditions expected to ease thereafter,...

https://ddnews.gov.in/en/heavy-rainfall-to-continue-over-northwest-india-gujarat-and-rajasthan-to-see-extended-showers-imd/

---------------------------------------------------------------------------------------------------------

In [121]:
from langchain.schema.runnable import RunnableLambda

def format_news(results):
    lines = []
    for result in results["news"]:
        lines.append(
            f"---News:{result['position']}---\n"
            f"{result.get('title','')}\n"
            f"🗓 {result.get('date','')}\n"
            f"📄 {result.get('snippet','')}\n"
            f"🔗 {result.get('link','')}\n"

            f"\n"
            f"--------------------------------------------------------------------------------------------------------------------"
            
        )
    return "\n".join(lines)

In [122]:
print(format_news(results))

---News:1---
Aug brings 21% seasonal rain to Gujarat; 15% in last week alone
🗓 11 hours ago
📄 Ahmedabad: In the first 24 days of Aug this year, the state received 21% of average seasonal rainfall. Out of the total, 15% was recorded in...
🔗 https://timesofindia.indiatimes.com/city/ahmedabad/aug-brings-21-seasonal-rain-to-gujarat-15-in-last-week-alone/articleshow/123511020.cms

--------------------------------------------------------------------------------------------------------------------
---News:2---
Heavy rainfall to continue over northwest India; Gujarat and Rajasthan to see extended showers: IMD
🗓 1 hour ago
📄 Northwest India is likely to experience heavy to very heavy rainfall until August 26, with conditions expected to ease thereafter,...
🔗 https://ddnews.gov.in/en/heavy-rainfall-to-continue-over-northwest-india-gujarat-and-rajasthan-to-see-extended-showers-imd/

-------------------------------------------------------------------------------------------------------------------

In [128]:
# Step 1L Creating an LLM

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

model_openai = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

model_gemini = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

In [133]:
# Step 2: Creating agent and react

from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

# Pull the react prompt from langchain hub

prompt = hub.pull("hwchase17/react")

C:\Users\jugal\langchain-env\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [134]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [138]:
# Step 4: Create an react_agent manually, with the pulled prompt

react_agent = create_react_agent(
    llm = model_gemini,
    tools = [serper_search],
    prompt = prompt
)

# Step 5: Wrap the agent withan Agent Executor

agent_executer = AgentExecutor(
    agent = react_agent,
    tools = [serper_search],
    verbose = True
)

AttributeError: 'GoogleSerperAPIWrapper' object has no attribute 'name'

`AttributeError: 'GoogleSerperAPIWrapper' object has no attribute 'name'` <br>
is because GoogleSerperAPIWrapper is a utility, not a Tool. <br>
LangChain agents expect Tool objects, which must have attributes like name and description.

In [203]:
from langchain.tools import Tool

# Create the search utility
serper_search = GoogleSerperAPIWrapper(type="search")
'''
# Wrap it into a Tool
search_tool = Tool(
    name="Google_Search",
    func=serper_search.results,
    description="Use this tool to search anything in Google Search Engine."
)
'''

'\n# Wrap it into a Tool\nsearch_tool = Tool(\n    name="Google_Search",\n    func=serper_search.results,\n    description="Use this tool to search anything in Google Search Engine."\n)\n'

In [204]:
from langchain_core.tools import tool

@tool
def search_tool(query:str) ->str:
    '''
    Use this tool to search for anything on Google Search Engine.
    Pass the query and it will return the results by searching on Google Search Engine.
    '''

    return serper_search.results(query)

GoogleSerperAPIWrapper is not a tool, it is a Wrapper as when we will pass its object directly to 'tools' argument,
it will give error because as a tool, it expects name and description.

So we will have to wrap it under a @tool decorator

In [205]:
# Step 4: Create an react_agent manually, with the pulled prompt

react_agent = create_react_agent(
    llm = model_gemini,
    tools = [search_tool],
    prompt = prompt
)

# Step 5: Wrap the agent withan Agent Executor

agent_executor = AgentExecutor(
    agent = react_agent,
    tools = [search_tool],
    handle_parsing_errors = True,
    verbose = True
)

In [206]:
response = agent_executor.invoke({'input': "Find best trains from Rajkot (Gujarat) to Vrindvan"})



> Entering new AgentExecutor chain...
Action: search_tool
Action Input: trains from Rajkot to Vrindavan{'searchParameters': {'q': 'trains from Rajkot to Vrindavan', 'gl': 'us', 'hl': 'en', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Rajkot To Vrindavan Trains | Book from 7 Trains - Goibibo', 'link': 'https://www.goibibo.com/trains/rajkot-to-vrindavan-trains/', 'snippet': '12475 - HAPA SVDK EXP is the fastest among the Rajkot to Vrindavan Trains. It departs from Rajkot Railway Station(RJT) at 09:38 AM and arrives at Rajkot Railway ...', 'position': 1}, {'title': 'Rajkot to Vrindavan Long-Distance Trains - Railway Enquiry', 'link': 'https://indiarailinfo.com/search/rjt-rajkot-junction-to-bdb-vrindavan/1570/0/5771', 'snippet': 'Fastest Train: Jamnagar - Shri Mata Vaishno Devi Katra SF Express (PT): 17h 5m: 1187 km. Slowest Train: Rajkot - Lalkuan Special Fare Summer Special: 23h 23m: ...', 'position': 2}, {'title': 'Rajkot To Mathura Trains | Book From 2 Tra

In [207]:
print(response['input'])
print()

print(response['output'])

Find best trains from Rajkot (Gujarat) to Vrindvan

The best trains from Rajkot to Vrindavan (via Mathura Junction) are:
1.  **HAPA SVDK EXP (Train No. 12475)**: Departs Rajkot (RJT) at 09:50 AM, arrives Mathura Junction (MTJ) at 03:10 AM (next day), taking approximately 17 hours 20 minutes. This train runs on **Tuesdays**.
2.  **JAM SVDK EXP (Train No. 12477)**: Departs Rajkot (RJT) at 10:05 AM, arrives Mathura Junction (MTJ) at 03:10 AM (next day), taking approximately 17 hours 05 minutes. This train runs on **Wednesdays**.


In [212]:
from langchain.agents import create_openai_functions_agent, AgentExecutor

prompt2 = hub.pull("hwchase17/openai-functions-agent")


functions_agent = create_openai_functions_agent(
    llm = model_openai,
    tools=[search_tool],
    prompt = prompt2
)

agent_executor2 = AgentExecutor(
    agent=functions_agent,
    tools=[search_tool],
    verbose=True
)

response2 = agent_executor2.invoke({'input': "What is the current whether in Rajkot(Gujarat)"})

C:\Users\jugal\langchain-env\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(




> Entering new AgentExecutor chain...

Invoking: `search_tool` with `{'query': 'current weather in Rajkot Gujarat'}`


{'searchParameters': {'q': 'current weather in Rajkot Gujarat', 'gl': 'us', 'hl': 'en', 'type': 'search', 'num': 10, 'engine': 'google'}, 'searchInformation': {'didYouMean': 'Google Weather'}, 'organic': [{'title': 'Rajkot, Gujarat, India Weather Forecast - AccuWeather', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/weather-forecast/202440', 'snippet': 'Rajkot, Gujarat, India Weather Forecast, with current conditions, wind, air quality, and what to expect for the next 3 days.', 'sitelinks': [{'title': 'Current Weather', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/current-weather/202440'}, {'title': 'Daily', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/daily-weather-forecast/202440'}, {'title': 'Hourly', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/hourly-weather-forecast/202440'}, {'title': 'Radar', 'link': 'https://www

In [213]:
print(response2['input'])
print()

print(response2['output'])

What is the current whether in Rajkot(Gujarat)

The current weather in Rajkot, Gujarat is 79°F, with cloudy skies. The RealFeel temperature is 87°F. The humidity is at 85%.


# 🧑‍💻 ReAct Agent vs Functions Agent
# ***🔹 ReAct Agent***

Works by prompting the LLM to "think step by step" (Reason + Act).

The model decides tool usage based on reasoning.

📝 Example (ReAct)

`User`: “What’s the weather in Delhi tomorrow?”

`Agent (Reasoning inside prompt):`

I need to check the weather API.
`Action:` call_weather_api(location="Delhi", date="tomorrow")

`Tool Response:` {"temp": "32°C", "condition": "Sunny"}

`Agent Final Answer:`
👉 “The weather in Delhi tomorrow will be sunny with 32°C.”

⚠️ Here the LLM freely decides when/how to call the tool. It might sometimes mess up tool names or inputs.

# ***🔹 Functions Agent***

Uses structured function calling.

The LLM doesn’t “hallucinate tools” — it outputs strict JSON matching a schema.

📝 Example (Functions)

`User:` “What’s the weather in Delhi tomorrow?”

`Agent Output (Structured JSON):`

{
  "name": "get_weather",  <br>
  "arguments": {  <br>
    "location": "Delhi",  <br>
    "date": "2025-08-27"  <br>
  }  <br>
}


`Tool Response:` {"temp": "32°C", "condition": "Sunny"}

`Agent Final Answer:`
👉 “The weather in Delhi tomorrow will be sunny with 32°C.”

✅ Much safer because the model can only pick from registered functions and fill in arguments.

# Which is used more today?

Functions Agent is used more in the real world (2025).

Companies choose it because it’s reliable, predictable, and works well with production apps.

ReAct is still great for prototyping where you want the model to show reasoning steps.

# ⚡ Simple rule:

Exploration / Research → ReAct

Production Apps → Functions

# Wrapping up everything

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import os
os.environ['SERPER_API_KEY'] = "0a4bee2f4a7701bc4dc73d4f0071260490e16094" # We can avoid this as we have added API key to .env file

from langchain_community.utilities import GoogleSerperAPIWrapper
serper_search = GoogleSerperAPIWrapper(type = 'search')

In [6]:
# Creating a search tool

from langchain_core.tools import tool

@tool
def search_tool(query:str) ->str:
    '''
    Use this tool to search for anything on Google Search Engine.
    Pass the query and it will return the results by searching on Google Search Engine.
    '''

    return serper_search.results(query)

GoogleSerperAPIWrapper is not a tool, it is a Wrapper as when we will pass its object directly to 'tools' argument,
it will give error because as a tool, it expects name and description.

So we will have to wrap it under a @tool decorator

In [10]:
# Step 1: Creating an LLM

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

model_openai = ChatOpenAI(
    model = 'gpt-3.5-turbo'
)

model_gemini = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

# Step 2: Creating prompt, function_agent and agent_executor

from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain import hub

prompt2 = hub.pull("hwchase17/openai-functions-agent")

function_agent = create_openai_functions_agent(
    llm = model_gemini,
    tools = [search_tool],
    prompt = prompt2
)

agent_executor = agent_executor2 = AgentExecutor(
    agent = function_agent,
    tools =[search_tool],
    verbose = True,
    handle_parsing_errors = True
)

C:\Users\jugal\langchain-env\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [18]:
response2 = agent_executor2.invoke({'input': "What is the whether today in Rajkot(Gujarat), in celcius"})



> Entering new AgentExecutor chain...

Invoking: `search_tool` with `{'query': 'weather in Rajkot, Gujarat in Celsius'}`


{'searchParameters': {'q': 'weather in Rajkot, Gujarat in Celsius', 'gl': 'us', 'hl': 'en', 'type': 'search', 'num': 10, 'engine': 'google'}, 'searchInformation': {'didYouMean': 'Google Weather'}, 'organic': [{'title': 'Rajkot, Gujarat, India Weather Forecast - AccuWeather', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/weather-forecast/202440', 'snippet': 'Rajkot, Gujarat, India Weather Forecast, with current conditions, wind, air quality, and what to expect for the next 3 days.', 'sitelinks': [{'title': 'Current Weather', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/current-weather/202440'}, {'title': 'Daily', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/daily-weather-forecast/202440'}, {'title': 'Weather Today for Rajkot...', 'link': 'https://www.accuweather.com/en/in/rajkot/202440/weather-today/202440'}, {'title': 'Hourly', 

In [19]:
print(response2['input'])
print()

print(response2['output'])

What is the whether today in Rajkot(Gujarat), in celcius

The current weather in Rajkot, Gujarat is 25°C with patchy rain nearby, according to one source. Another source indicates it is 37.4°C and partly cloudy.


# That's all, such simple is to create an AI Agent.

# Now to to another notebook `Part 2` where we will also add a tool to get the live whether details of any city